# Notebook 02 — Family A Parser Prototype

This notebook prototyped the parser for single-value worksheets such as Cours, Bid, Ask, Volume MC and Quantité MC.

## Design rule

The Family A parser is responsible for raw market data sheets only. It should reshape the sheet into a long table where each row is (Date, CODE ISIN, Variable, Value).

In [4]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'samples').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current notebook location.')

repo_root = find_repo_root(Path.cwd().resolve())
workbook_path = repo_root / 'samples' / 'Données Marché Boursier_Projet_IA_copy.xlsx'
print('Workbook exists:', workbook_path.exists())

Workbook exists: True


In [ ]:
raw = pd.read_excel(workbook_path, sheet_name='Cours', header=None)

# Show a compact, readable preview of the sheet
preview = raw.iloc[:12, :8].copy()
preview.columns = [f'Col {i}' for i in range(preview.shape[1])]
preview

,Col 0,Col 1,Col 2,Col 3,Col 4,Col 5,Col 6,Col 7
0,Code AMC,1229,1211,1095,1094,1258,1181,1093
1,CODE ISIN,MA0000012296,MA0000012114,MA0000010951,MA0000010944,MA0000012585,MA0000011819,MA0000010936
2,LIBELLE,AFMA,AFRIC INDUSTRIES SA,AFRIQUIA GAZ,AGMA,AKDITAL,ALLIANCES,ALUMINIUM DU MAROC
3,NaN,"MA0000012296,XX,CAS","MA0000012114,XX,CAS","MA0000010951,XX,CAS","MA0000010944,XX,CAS","MA0000012585,XX,CAS","MA0000011819,XX,CAS","MA0000010936,XX,CAS"
4,NaN,AFMA P,Afric Indus.,Afriquia Gaz P,Agma P,Akdital Br,Alliances P,Aluminium Maroc P
5,NaN,VAL,VAL,VAL,VAL,VAL,VAL,VAL
6,2018-12-31 00:00:00,990,270,3000,3079,NaN,85,1565
7,2019-01-02 00:00:00,990,270,3000,3079,NaN,86.5,1658
8,2019-01-03 00:00:00,990,286,2915,3079,NaN,84,1698
9,2019-01-04 00:00:00,980,286,2940,3079,NaN,76.51,1698


## Parser responsibilities

1. Detect metadata rows such as CODE ISIN and LIBELLE.
2. Detect the row that contains trading dates.
3. Map each company column to its CODE ISIN and company name.
4. Reshape the sheet into a long table that can later be merged with other raw market sheets.